# TranSTR + DINOv3 + NCOD LUM+HUM — Colab Pro
**Khác bản Kaggle**: data qua `kaggle_hub`, bs=32, no gradient accumulation, 10 epochs, EMA weights cho eval/save, tắt runtime cell cuối.

In [ ]:
# CELL 0: Mount Google Drive (lưu checkpoint)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL  = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH    = 'hum'

if not os.path.exists(REPO_NAME):
    print(f'Cloning {REPO_URL}...')
    os.system(f'git clone {REPO_URL} -b {BRANCH}')
else:
    print('Repo already exists.')

target_dir = os.path.join(os.getcwd(), REPO_NAME, 'causalvid')
if os.path.exists(target_dir) and os.path.basename(os.getcwd()) != 'causalvid':
    os.chdir(target_dir)
    print(f'Changed directory to: {os.getcwd()}')
else:
    print(f'Working directory: {os.getcwd()}')

In [ ]:
# CELL 2: Install dependencies & W&B login
print('=== CELL 2: Install & W&B Setup ===')
os.system('pip install -q wandb kaggle kaggle-hub --upgrade')

import wandb

# ============================================
# W&B CONFIG
# ============================================
WANDB_API_KEY = ''  # 🔴 API key
WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY  = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in!')

In [ ]:
# CELL 3: Setup Kaggle API key & download datasets via kaggle_hub
print('=== CELL 3: Kaggle Hub Data Download ===')

# ============================================
# 🔴 KAGGLE API CREDENTIALS
# Option A: Dùng Colab Secrets (khuyến nghị)
#   1. Vào Colab > Secrets (icon 🔑 bên trái)
#   2. Thêm KAGGLE_USERNAME và KAGGLE_KEY
# Option B: Upload kaggle.json thủ công
# ============================================
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✅ Kaggle credentials loaded from Colab Secrets')
except Exception as e:
    print(f'⚠️  Could not load from Secrets: {e}')
    print('   Trying kaggle.json fallback...')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    # Nếu đã upload kaggle.json thủ công thì bỏ qua

import kaggle_hub

# ============================================
# Dataset slugs — giữ nguyên IDs từ Kaggle notebook
# ============================================
print('\nDownloading datasets (cached nếu đã có)...')

# DINOv3 features (train/val/test split)
DINOV3_HUB_PATH   = kaggle_hub.dataset_download('danielq07/dinov3-feat')
print(f'✅ DINOv3 features: {DINOV3_HUB_PATH}')

# Object detection features
OBJ_HUB_PATH      = kaggle_hub.dataset_download('danielq07/object-detection-causal-full')
print(f'✅ Object features: {OBJ_HUB_PATH}')

# Text annotations
ANN_HUB_PATH      = kaggle_hub.dataset_download('danielq07/text-annotation')
print(f'✅ Annotations: {ANN_HUB_PATH}')

# Splits
SPLIT_HUB_PATH    = kaggle_hub.dataset_download('danielq07/casual-vid-data-split')
print(f'✅ Splits: {SPLIT_HUB_PATH}')

In [ ]:
# CELL 4: Merge DINOv3 features (train/val/test → merged folder)
print('=== CELL 4: Merge DINOv3 Features ===')
import shutil
from tqdm.auto import tqdm

CLIP_MERGED_PATH = '/content/dinov3_T16_dim1024_merge'
os.makedirs(CLIP_MERGED_PATH, exist_ok=True)

# Auto-detect source với train/val/test subfolders
candidate_roots = [
    DINOV3_HUB_PATH,
    os.path.join(DINOV3_HUB_PATH, 'features'),
]

def has_split_subdirs(root):
    if not os.path.exists(root):
        return False
    names = set(os.listdir(root))
    return ('train' in names) and ('test' in names) and ('valid' in names or 'val' in names)

resolved_source = None
for cand in candidate_roots:
    if has_split_subdirs(cand):
        resolved_source = cand
        break

if resolved_source is None:
    raise FileNotFoundError(f'Không tìm thấy thư mục DINOv3. Đã thử: {candidate_roots}')

print(f'Source: {resolved_source}')
total_copied = 0
for split in ['train', 'test', 'valid', 'val']:
    split_folder = os.path.join(resolved_source, split)
    if not os.path.exists(split_folder):
        continue
    pt_files = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
    print(f'📁 {split}: {len(pt_files)} files')
    for fname in tqdm(pt_files, desc=f'Copying {split}'):
        src = os.path.join(split_folder, fname)
        dst = os.path.join(CLIP_MERGED_PATH, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            total_copied += 1

final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
print(f'\n✅ Merge complete! Total .pt: {final_count} (copied {total_copied} mới)')

In [ ]:
# CELL 5: Imports
print('=== CELL 5: Imports ===')
import torch, numpy as np, pandas as pd, json
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.util import set_seed
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports OK')

In [ ]:
# CELL 6: Train/Eval functions — NCOD LUM+HUM (no gradient accumulation)
print('=== CELL 6: Functions (NCOD LUM + HUM) ===')

NUM_CHOICES = 5


def train_epoch_ncod_hum(
    model, opt_model, opt_U, U, loader, device, epoch,
    ema_model=None, hum_alpha=1.0
):
    """Bi-level NCOD training với HUM — không dùng gradient accumulation (accumulation_steps=1).

    LUM (Descriptive/Explanatory): NCOD shifted-CE giảm trọng số mẫu nhiễu.
    HUM (Predictive/Counterfactual + Reason): (1 + alpha*U) * CE tăng penalty khi U cao.
    EMA: nếu ema_model được truyền vào, update sau mỗi batch.
    """
    model.train()
    total_l1 = total_l1_lum = total_l1_hum = 0.0
    total_l2 = 0.0
    correct = total = 0
    n_hard_total = n_easy_total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, sample_indices = batch
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
        sample_indices = sample_indices.long()

        is_hard = torch.tensor(
            ['predictive' in k or 'counterfactual' in k for k in qns_keys],
            dtype=torch.bool, device=device
        )  # [B]

        # Forward
        logits = model(ff, of, q, a)              # [B, 5]
        probs  = F.softmax(logits, dim=1)         # [B, 5]

        u_batch  = U[sample_indices].unsqueeze(1) # [B, 1]
        y_onehot = F.one_hot(tgt, num_classes=NUM_CHOICES).float()  # [B, 5]

        # ─── L1: update model ───────────────────────────────────────────
        ce_per_sample = -torch.sum(
            y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1
        )  # [B]

        # LUM branch: NCOD shifted-CE
        shifted_probs = probs + (u_batch.detach() * y_onehot)
        shifted_probs = torch.clamp(shifted_probs, min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)  # [B]

        # HUM branch: UCT penalty scaling
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample  # [B]

        # Fuse: HUM cho hard samples, LUM cho easy samples
        fused = torch.where(is_hard, hum_loss, lum_loss)  # [B]
        L1 = fused.mean()

        n_hard = int(is_hard.sum().item())
        n_easy = int((~is_hard).sum().item())
        l1_hum_val = hum_loss[is_hard].mean().item()  if n_hard > 0 else 0.0
        l1_lum_val = lum_loss[~is_hard].mean().item() if n_easy > 0 else 0.0

        # Backward L1 (no scaling — accumulation_steps=1)
        opt_model.zero_grad()
        L1.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt_model.step()

        # ─── L2: update U ───────────────────────────────────────────────
        probs_det   = probs.detach()
        shifted_det = probs_det + (u_batch * y_onehot)
        L2 = F.mse_loss(shifted_det, y_onehot)

        opt_U.zero_grad()
        L2.backward()
        opt_U.step()
        with torch.no_grad():
            U.clamp_(0.0, 0.99)

        # EMA update after each batch
        if ema_model is not None:
            ema_model.update_parameters(model)

        # Tracking
        total_l1     += L1.item()
        total_l1_lum += l1_lum_val
        total_l1_hum += l1_hum_val
        total_l2     += L2.item()
        n_hard_total += n_hard
        n_easy_total += n_easy
        correct += (logits.argmax(-1) == tgt).sum().item()
        total   += tgt.size(0)

        pbar.set_postfix({
            'L1': total_l1 / (batch_idx + 1),
            'LUM': total_l1_lum / (batch_idx + 1),
            'HUM': total_l1_hum / (batch_idx + 1),
            'L2':  total_l2 / (batch_idx + 1),
            'acc': correct / total * 100
        })

        if batch_idx % 50 == 0:
            wandb.log({
                'batch_L1':     L1.item(),
                'batch_L1_LUM': l1_lum_val,
                'batch_L1_HUM': l1_hum_val,
                'batch_L2':     L2.item(),
                'batch_n_hard': n_hard,
                'batch_n_easy': n_easy,
                'batch_acc': (logits.argmax(-1) == tgt).float().mean().item() * 100,
                'batch': epoch * len(loader) + batch_idx
            })

    n = len(loader)
    acc = correct / total * 100
    print(f'  [Epoch {epoch}] hard={n_hard_total} easy={n_easy_total} '
          f'ratio_hard={n_hard_total/(n_hard_total+n_easy_total)*100:.1f}%')
    return total_l1/n, total_l1_lum/n, total_l1_hum/n, total_l2/n, acc


def eval_epoch(model, loader, device):
    """Evaluation — không dùng U."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _qns_key, _sample_indices = batch
            out = model(ff.to(device), of.to(device), q, a)
            correct += (out.argmax(-1) == ans_id.to(device)).sum().item()
            total   += ans_id.size(0)
    return correct / total * 100


print('Functions defined (NCOD LUM+HUM, no accumulation)!')

In [ ]:
# CELL 7: Paths & Config
print('=== CELL 7: Paths & Config ===')

# Paths từ kaggle_hub download (CELL 3)
OBJ_FEATURE_PATH = OBJ_HUB_PATH
ANNOTATION_PATH  = os.path.join(ANN_HUB_PATH, 'QA')
SPLIT_DIR        = os.path.join(SPLIT_HUB_PATH, 'split')

# Nếu subfolder khác tên, auto-detect
if not os.path.exists(ANNOTATION_PATH):
    ANNOTATION_PATH = ANN_HUB_PATH
if not os.path.exists(SPLIT_DIR):
    SPLIT_DIR = SPLIT_HUB_PATH

BASE      = '/content/drive/MyDrive/transtr_colab'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'✅ {name}: {items}')
        return True
    print(f'❌ {name}: NOT FOUND - {path}')
    return False

all_ok  = verify_path('DINOv3 Features (merged)', CLIP_MERGED_PATH)
all_ok &= verify_path('Object Features', OBJ_FEATURE_PATH)
all_ok &= verify_path('Annotations', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)
if not all_ok:
    print('\n⚠️ Kiểm tra lại dataset slugs ở CELL 3!')

RUN_TRAINING   = True
MODEL_FILENAME = 'best_model_dinov3_ncod_hum_colab.ckpt'
FEAT_DIM       = 1024


class Config:
    # Paths
    video_feature_root  = CLIP_MERGED_PATH
    object_feature_path = OBJ_FEATURE_PATH
    sample_list_path    = ANNOTATION_PATH
    split_dir_txt       = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs       = 20
    frames     = 16
    select_frames = 5
    topK_obj   = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim   = 2053
    d_model        = 768
    word_dim       = 768
    nheads         = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before   = True
    activation = 'gelu'
    dropout         = 0.3
    encoder_dropout  = 0.3

    # Text encoder
    text_encoder_type   = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr     = 5e-6
    text_pool_mode      = 0

    # ============ COLAB CHANGES ============
    bs                 = 32          # batch size (Kaggle: 8)
    accumulation_steps = 1           # tắt gradient accumulation (Kaggle: 4)
    epoch              = 10          # số epochs (Kaggle: biến động)
    # =======================================

    lr           = 1e-5
    decay        = 1e-4
    gamma        = 0.5
    patience     = 3
    n_query      = 5
    num_workers  = 4
    hard_eval    = False

    # NCOD
    ncod_u_lr        = 0.01
    ncod_u_mean      = 0.5
    ncod_u_std       = 0.1
    ncod_u_clamp_max = 0.99

    # HUM
    hum_alpha = 1.0

    # EMA
    ema_decay = 0.999


args = Config()
print(f'\n🔧 Config:')
print(f'  bs={args.bs} | accum={args.accumulation_steps} | effective_bs={args.bs} | epochs={args.epoch}')
print(f'  EMA decay={args.ema_decay} | HUM alpha={args.hum_alpha}')
print(f'  Backbone: DINOv3 (D={FEAT_DIM})')

In [ ]:
# CELL 8: Create Datasets & DataLoaders
print('=== CELL 8: Datasets ===')

print('\n--- Creating TRAIN dataset ---')
train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True
)
print('\n--- Creating VAL dataset ---')
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True
)
print('\n--- Creating TEST dataset ---')
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True
)

if len(train_ds) == 0:
    raise ValueError(f'Train dataset rỗng! video_feature_root={args.video_feature_root}')
if len(val_ds)   == 0: print('⚠️ Val dataset rỗng.')
if len(test_ds)  == 0: print('⚠️ Test dataset rỗng.')

# Lưu sample keys để phân tích U sau này
train_sample_keys = [train_ds.sample_list.iloc[i]['video_id'] + '_' +
                     train_ds.sample_list.iloc[i]['type']
                     for i in range(len(train_ds))]

train_loader = DataLoader(train_ds, args.bs, shuffle=True,
                          num_workers=args.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   args.bs, shuffle=False,
                          num_workers=args.num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  args.bs, shuffle=False,
                          num_workers=args.num_workers, pin_memory=True)

print(f'\n✅ Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'   Batches/epoch: {len(train_loader)}')

In [ ]:
# CELL 9: Model + NCOD U + Optimizers + EMA
print('=== CELL 9: Model + NCOD + EMA ===')

cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device']     = device
cfg['topK_frame'] = args.select_frames

print(f'Creating model with frame_feat_dim = {cfg["frame_feat_dim"]}')
model = VideoQAmodel(**cfg)
model.to(device)

# NCOD: Khởi tạo U
num_train_samples = len(train_ds)
print(f'\n🔧 NCOD: Initializing U [{num_train_samples}]')
U = torch.nn.Parameter(
    torch.abs(torch.randn(num_train_samples, device=device) * args.ncod_u_std + args.ncod_u_mean)
)

# EMA model (torch built-in, decay=0.999)
from torch.optim.swa_utils import AveragedModel
ema_model = AveragedModel(
    model,
    multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(args.ema_decay)
)
print(f'✅ EMA model initialized (decay={args.ema_decay})')

# Optimizer 1: Model
opt_model = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters()
                if 'text_encoder' not in n and p.requires_grad]},
    {'params': [p for n, p in model.named_parameters()
                if 'text_encoder' in n and p.requires_grad],
     'lr': args.text_encoder_lr}
], lr=args.lr, weight_decay=args.decay)

# Optimizer 2: U
opt_U = torch.optim.SGD([U], lr=args.ncod_u_lr)

scheduler = ReduceLROnPlateau(opt_model, 'max', factor=args.gamma, patience=args.patience)
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M')
print(f'U: shape=[{num_train_samples}]')
print(f'HUM alpha: {args.hum_alpha}')

In [ ]:
# CELL 10: Resume Logic
print('=== CELL 10: Resume Logic ===')

# ============================================
# 🔴 Set RESUME = True để tiếp tục từ checkpoint
# ============================================
RESUME               = False
WANDB_RESUME_ARTIFACT = ''  # e.g. 'your-entity/transtr-causalvid-dino/checkpoint:latest'

start_epoch = 1
best_acc    = 0
history     = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'ema_val_acc': []}

if RESUME:
    resume_path = None
    latest_local = os.path.join(MODEL_DIR, 'latest_checkpoint_dinov3_ncod_hum_colab.ckpt')

    if os.path.exists(latest_local):
        resume_path = latest_local
        print(f'📁 Found local latest: {latest_local}')
    elif os.path.exists(save_path):
        resume_path = save_path
        print(f'📁 Found local best: {save_path}')
    elif WANDB_RESUME_ARTIFACT:
        print(f'☁️ Downloading from W&B: {WANDB_RESUME_ARTIFACT}')
        try:
            api = wandb.Api()
            artifact = api.artifact(WANDB_RESUME_ARTIFACT)
            artifact_dir = artifact.download(root=MODEL_DIR)
            for root_dir, _dirs, files in os.walk(artifact_dir):
                for fname in files:
                    if fname.endswith('.ckpt'):
                        resume_path = os.path.join(root_dir, fname)
                        break
                if resume_path:
                    break
            if resume_path:
                print(f'✅ Downloaded: {resume_path}')
        except Exception as e:
            print(f'❌ W&B download failed: {e}')

    if resume_path and os.path.exists(resume_path):
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt.get('model_state_dict', ckpt))
        if 'ema_state_dict' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            opt_model.load_state_dict(ckpt['optimizer_state_dict'])
        if 'U' in ckpt:
            U.data = ckpt['U'].to(device)
        start_epoch = ckpt.get('epoch', 0) + 1
        best_acc    = ckpt.get('best_acc', 0)
        history     = ckpt.get('history', history)
        print(f'✅ Resumed from epoch {start_epoch-1}, best_acc={best_acc:.2f}%')
    else:
        print('ℹ️ No checkpoint found, starting fresh.')
else:
    print('ℹ️ RESUME=False, starting fresh.')

In [ ]:
# CELL 11: Initialize W&B Run
print('=== CELL 11: W&B Init ===')

wandb_config = {
    'model':             'TranSTR',
    'backbone':          'openai/dinov3_T16_dim1024',
    'variant':           'NCOD_LUM+HUM+EMA',
    'frame_feat_dim':    args.frame_feat_dim,
    'text_encoder':      args.text_encoder_type,
    'batch_size':        args.bs,
    'accumulation_steps': 1,
    'effective_batch_size': args.bs,
    'learning_rate':     args.lr,
    'epochs':            args.epoch,
    'd_model':           args.d_model,
    'nheads':            args.nheads,
    'num_encoder_layers': args.num_encoder_layers,
    'dropout':           args.dropout,
    'topK_frame':        args.select_frames,
    'topK_obj':          args.topK_obj,
    'train_samples':     len(train_ds),
    'val_samples':       len(val_ds),
    'ncod_enabled':      True,
    'hum_enabled':       True,
    'ncod_u_lr':         args.ncod_u_lr,
    'ncod_u_clamp_max':  args.ncod_u_clamp_max,
    'hum_alpha':         args.hum_alpha,
    'ema_enabled':       True,
    'ema_decay':         args.ema_decay,
    'platform':          'colab_pro',
}

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config=wandb_config,
    name=f'colab-transtr-dinov3-ncod-hum-ema-ep{args.epoch}-bs{args.bs}',
    reinit=True
)
wandb.watch(model, log='gradients', log_freq=100)
print(f'✅ W&B Run: {run.url}')

In [ ]:
# CELL 12: Training Loop (NCOD LUM+HUM + EMA)
print('=== CELL 12: Training ===')

if 'start_epoch' not in dir(): start_epoch = 1
if 'best_acc'    not in dir(): best_acc    = 0
if 'history'     not in dir():
    history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'ema_val_acc': []}

print(f'📌 Start epoch={start_epoch} | best_acc={best_acc:.2f}%')

if RUN_TRAINING:
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')

        avg_l1, avg_l1_lum, avg_l1_hum, avg_l2, train_acc = train_epoch_ncod_hum(
            model, opt_model, opt_U, U, train_loader, device, ep,
            ema_model=ema_model, hum_alpha=args.hum_alpha
        )

        # Eval với raw model
        val_acc = eval_epoch(model, val_loader, device)

        # Eval với EMA model (thường tốt hơn)
        ema_val_acc = eval_epoch(ema_model, val_loader, device)

        # Scheduler step dựa trên EMA val acc
        scheduler.step(ema_val_acc)

        history['train_loss'].append(avg_l1)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['ema_val_acc'].append(ema_val_acc)

        # U statistics
        u_np = U.detach().cpu().numpy()
        u_by_type = {}
        for i, key in enumerate(train_sample_keys):
            for suffix in ['predictive_reason', 'counterfactual_reason',
                           'predictive', 'counterfactual',
                           'descriptive', 'explanatory']:
                if key.endswith('_' + suffix):
                    u_by_type.setdefault(suffix, []).append(u_np[i])
                    break
        u_qtype_log = {f'U/{qt}/mean': float(np.mean(vals))
                       for qt, vals in u_by_type.items()}

        log_dict = {
            'epoch':          ep,
            'train_L1':       avg_l1,
            'train_L1_LUM':   avg_l1_lum,
            'train_L1_HUM':   avg_l1_hum,
            'train_L2':       avg_l2,
            'train_acc':      train_acc,
            'val_acc':        val_acc,
            'ema_val_acc':    ema_val_acc,
            'U/mean':         float(u_np.mean()),
            'U/std':          float(u_np.std()),
            'lr':             opt_model.param_groups[0]['lr'],
            **u_qtype_log
        }
        wandb.log(log_dict)

        print(f'  Train acc={train_acc:.2f}% | Val acc={val_acc:.2f}% | EMA Val acc={ema_val_acc:.2f}%')
        print(f'  L1={avg_l1:.4f} (LUM={avg_l1_lum:.4f} HUM={avg_l1_hum:.4f}) | L2={avg_l2:.4f}')

        # Save best checkpoint (dùng EMA val acc)
        if ema_val_acc > best_acc:
            best_acc = ema_val_acc
            ckpt = {
                'epoch':                ep,
                'model_state_dict':     model.state_dict(),
                'ema_state_dict':       ema_model.state_dict(),
                'optimizer_state_dict': opt_model.state_dict(),
                'U':                    U.detach().cpu(),
                'best_acc':             best_acc,
                'history':              history,
            }
            torch.save(ckpt, save_path)
            print(f'  💾 Best EMA checkpoint saved (EMA val={best_acc:.2f}%)')
            # W&B artifact
            art = wandb.Artifact('best-ema-checkpoint-dinov3-ncod-hum', type='model')
            art.add_file(save_path)
            run.log_artifact(art)

        # Save latest checkpoint mỗi epoch
        latest_path = os.path.join(MODEL_DIR, 'latest_checkpoint_dinov3_ncod_hum_colab.ckpt')
        ckpt_latest = {
            'epoch':                ep,
            'model_state_dict':     model.state_dict(),
            'ema_state_dict':       ema_model.state_dict(),
            'optimizer_state_dict': opt_model.state_dict(),
            'U':                    U.detach().cpu(),
            'best_acc':             best_acc,
            'history':              history,
        }
        torch.save(ckpt_latest, latest_path)

print(f'\n✅ Training done! Best EMA val acc = {best_acc:.2f}%')

In [ ]:
# CELL 13: Detailed Evaluation
print('=== CELL 13: Detailed Evaluation ===')
import seaborn as sns

def evaluate_detailed(eval_model, loader, device, log_to_wandb=True):
    """Tính metric chi tiết theo question type."""
    eval_model.eval()
    type_results = {}

    print('\n📊 Running Detailed Evaluation...')
    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, _sample_indices = batch
            ff, of = ff.to(device), of.to(device)
            out = eval_model(ff, of, qns, ans_word)
            preds   = out.argmax(dim=-1).cpu().numpy()
            targets = ans_id.numpy()

            for key, pred, target in zip(qns_keys, preds, targets):
                qtype = 'unknown'
                for suffix in ['predictive_reason', 'counterfactual_reason',
                               'predictive', 'counterfactual',
                               'descriptive', 'explanatory']:
                    if ('_' + suffix) in key:
                        qtype = suffix
                        break
                type_results.setdefault(qtype, []).append({
                    'pred': int(pred), 'target': int(target),
                    'correct': int(pred) == int(target)
                })

    metrics = {}
    print('\n📊 Accuracy by Question Type:')
    print('-' * 40)
    for qtype, results in sorted(type_results.items()):
        acc = sum(r['correct'] for r in results) / len(results) * 100
        metrics[qtype] = {'acc': acc, 'count': len(results)}
        print(f'  {qtype:30s}: {acc:5.2f}%  (n={len(results)})')

    all_correct = sum(r['correct'] for rs in type_results.values() for r in rs)
    all_total   = sum(len(rs) for rs in type_results.values())
    overall_acc = all_correct / all_total * 100
    metrics['overall'] = {'acc': overall_acc, 'count': all_total}
    print(f'  {"OVERALL":30s}: {overall_acc:5.2f}%  (n={all_total})')

    if log_to_wandb:
        wandb.log({'eval/' + k: v['acc'] for k, v in metrics.items()})
    return metrics

# Load best EMA checkpoint để eval cuối
if os.path.exists(save_path):
    ckpt = torch.load(save_path, map_location=device)
    ema_model.load_state_dict(ckpt['ema_state_dict'])
    print(f'Loaded best EMA checkpoint (epoch={ckpt["epoch"]}, best_acc={ckpt["best_acc"]:.2f}%)')

print('\n--- Test set evaluation (EMA model) ---')
metrics = evaluate_detailed(ema_model, test_loader, device, log_to_wandb=True)

In [ ]:
# CELL 14: Finish W&B + Tắt Runtime Colab
print('=== CELL 14: Finish & Shutdown ===')

# Save final metrics
with open('final_metrics_dinov3_ncod_hum_colab.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: final_metrics_dinov3_ncod_hum_colab.json')

# Log final artifact
final_artifact = wandb.Artifact(
    name='final-results-dinov3-ncod-hum-colab',
    type='results',
    metadata={'platform': 'colab_pro', 'ema': True, 'bs': args.bs, 'epochs': args.epoch}
)
final_artifact.add_file('final_metrics_dinov3_ncod_hum_colab.json')
if os.path.exists(save_path):
    final_artifact.add_file(save_path)
wandb.log_artifact(final_artifact)

wandb.finish()
print('✅ W&B run finished!')

# Ngắt runtime Colab để không tốn compute
from google.colab import runtime
runtime.unassign()